# Mobile Money Fraud Detection — Data Cleaning & EDA

This notebook performs the data cleaning and exploratory data analysis (EDA) for the PaySim mobile money fraud detection project.

### Project focus
The final model will use **Isolation Forest**, an unsupervised anomaly-detection algorithm.

The initial proposal features are:

- `type`
- `amount`
- `oldbalanceOrg`
- `newbalanceOrig`

The `isFraud` column is **not a model-training feature**. It is retained only to understand known fraud patterns during EDA and later evaluate the unsupervised model.

### Notebook workflow

1. Load the full PaySim dataset
2. Inspect data quality
3. Clean and validate the data
4. Examine fraud prevalence
5. Analyze transaction types
6. Compare normal and known fraudulent transactions
7. Study origin-account balance behavior
8. Examine feature relationships and outliers
9. Engineer EDA-informed behavioral features
10. Summarize findings for later modeling


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Load the full PaySim dataset

The project copy of the dataset is expected at `data/dataset.csv`.

If this notebook is run from the `notebooks/` folder, `../data/dataset.csv` is also checked automatically.


In [ ]:
candidate_paths = [
    Path("data/dataset.csv"),
    Path("../data/dataset.csv"),
    Path(r"D:\COURSES\Artificial Inteligence\Dataset\archive (1)\PS_20174392719_1491204439457_log.csv"),
]

file_path = next((path for path in candidate_paths if path.exists()), None)

if file_path is None:
    raise FileNotFoundError(
        "PaySim dataset not found. Place the full dataset at data/dataset.csv "
        "or update candidate_paths."
    )

print(f"Loading dataset from: {file_path}")

dataset = pd.read_csv(file_path)

print("\nDataset loaded successfully.")
print(f"Rows: {dataset.shape[0]:,}")
print(f"Columns: {dataset.shape[1]}")

display(dataset.head())


## 2. Initial data-quality inspection

Before changing anything, inspect:

- column names and data types
- missing values
- duplicate rows
- invalid negative transaction values
- constant columns

No outliers are removed here because unusual transactions may be important for anomaly detection.


In [ ]:
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(f"Rows: {dataset.shape[0]:,}")
print(f"Columns: {dataset.shape[1]}")
print("\nColumns:")
print(dataset.columns.tolist())

print("\nDATA TYPES")
print("-" * 70)
print(dataset.dtypes)

missing_summary = pd.DataFrame({
    "MissingValues": dataset.isnull().sum(),
    "MissingPercentage": (dataset.isnull().mean() * 100).round(4),
})

print("\nMISSING VALUES")
print("-" * 70)
display(missing_summary)

duplicate_count = dataset.duplicated().sum()

print("\nDUPLICATE ROWS")
print("-" * 70)
print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_count / len(dataset) * 100:.4f}%")

print("\nVALUE VALIDITY CHECK")
print("-" * 70)
print("Negative transaction amounts:", (dataset["amount"] < 0).sum())
print("Negative old origin balances:", (dataset["oldbalanceOrg"] < 0).sum())
print("Negative new origin balances:", (dataset["newbalanceOrig"] < 0).sum())

constant_columns = [
    column for column in dataset.columns
    if dataset[column].nunique(dropna=False) <= 1
]

print("\nCONSTANT COLUMNS")
print("-" * 70)
print(constant_columns)


## 3. Data cleaning

The full PaySim dataset contains no missing values or exact duplicates in our inspection, but the cleaning operations are kept in the notebook so the workflow remains reproducible.

The four baseline model features are preserved separately.

`isFraud` remains outside the model feature table and will later be used only for evaluation.


In [ ]:
cleaned_data = dataset.copy()

required_columns = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "isFraud",
]

cleaned_data = cleaned_data.drop_duplicates()
cleaned_data = cleaned_data.dropna(subset=required_columns)

feature_columns = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
]

model_data = cleaned_data[feature_columns].copy()
fraud_labels = cleaned_data["isFraud"].copy()

print("=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

print(f"Original rows: {len(dataset):,}")
print(f"Clean rows: {len(cleaned_data):,}")
print(f"Rows removed: {len(dataset) - len(cleaned_data):,}")

print("\nSelected baseline features:")
print(feature_columns)

print("\nMissing values in model data:")
print(model_data.isnull().sum())


# Exploratory Data Analysis


## 4. Fraud distribution

PaySim is highly imbalanced. This is important because normal accuracy can be misleading in fraud detection; later evaluation should emphasize precision, recall, F1-score, and false positives.


In [ ]:
total_transactions = len(cleaned_data)
fraud_count = (cleaned_data["isFraud"] == 1).sum()
normal_count = (cleaned_data["isFraud"] == 0).sum()

print(f"Total transactions: {total_transactions:,}")
print(f"Normal transactions: {normal_count:,}")
print(f"Fraud transactions: {fraud_count:,}")
print(f"Normal percentage: {normal_count / total_transactions * 100:.4f}%")
print(f"Fraud percentage: {fraud_count / total_transactions * 100:.4f}%")


## 5. Transaction-type analysis

This checks whether known fraudulent transactions are concentrated in specific transaction types.


In [ ]:
type_analysis = (
    cleaned_data
    .groupby("type")
    .agg(
        Transactions=("type", "size"),
        FraudTransactions=("isFraud", "sum"),
        AverageAmount=("amount", "mean"),
    )
)

type_analysis["TransactionPercentage"] = (
    type_analysis["Transactions"] / total_transactions * 100
)

type_analysis["FraudRate"] = (
    type_analysis["FraudTransactions"]
    / type_analysis["Transactions"]
    * 100
)

display(
    type_analysis
    .sort_values("Transactions", ascending=False)
    .round(4)
)


In [ ]:
type_counts = cleaned_data["type"].value_counts()

plt.figure(figsize=(9, 5))
plt.bar(type_counts.index, type_counts.values)
plt.title("Transactions by Type")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Transaction amount analysis

The full amount distribution is strongly right-skewed, so a log-transformed view is used for visualization only. No high-value records are removed.


In [ ]:
display(
    cleaned_data["amount"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

plt.figure(figsize=(9, 5))
plt.hist(np.log1p(cleaned_data["amount"]), bins=60)
plt.title("Distribution of Transaction Amounts")
plt.xlabel("log(Amount + 1)")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


## 7. Normal vs known fraud

The label is used here only to understand patterns already present in the dataset. It is not used as an Isolation Forest training input.


In [ ]:
fraud_comparison = (
    cleaned_data
    .groupby("isFraud")
    .agg(
        Transactions=("isFraud", "size"),
        AvgAmount=("amount", "mean"),
        MedianAmount=("amount", "median"),
        AvgOldBalance=("oldbalanceOrg", "mean"),
        MedianOldBalance=("oldbalanceOrg", "median"),
        AvgNewBalance=("newbalanceOrig", "mean"),
        MedianNewBalance=("newbalanceOrig", "median"),
    )
)

display(fraud_comparison.round(2))


## 8. Origin-account balance behavior

EDA showed that the relationship between transaction amount and sender balance is more informative than looking only at raw values.


In [ ]:
cleaned_data["originBalanceChange"] = (
    cleaned_data["oldbalanceOrg"]
    - cleaned_data["newbalanceOrig"]
)

cleaned_data["originEmptied"] = (
    cleaned_data["newbalanceOrig"] == 0
).astype(int)

balance_analysis = (
    cleaned_data
    .groupby("isFraud")
    .agg(
        AvgBalanceBefore=("oldbalanceOrg", "mean"),
        AvgBalanceAfter=("newbalanceOrig", "mean"),
        AvgBalanceChange=("originBalanceChange", "mean"),
        MedianBalanceChange=("originBalanceChange", "median"),
        AccountsEndingAtZero=("originEmptied", "sum"),
        PercentageEndingAtZero=("originEmptied", lambda x: x.mean() * 100),
    )
)

display(balance_analysis.round(4))


## 9. Amount relative to origin balance


In [ ]:
cleaned_data["amountToBalanceRatio"] = np.where(
    cleaned_data["oldbalanceOrg"] > 0,
    cleaned_data["amount"] / cleaned_data["oldbalanceOrg"],
    0,
)

ratio_analysis = (
    cleaned_data
    .groupby("isFraud")["amountToBalanceRatio"]
    .agg(["count", "mean", "median", "min", "max"])
)

display(ratio_analysis.round(4))


## 10. Numerical feature correlation

Very strong correlation can indicate redundant raw information. In this dataset, `oldbalanceOrg` and `newbalanceOrig` are expected to be highly correlated.


In [ ]:
correlation_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
]

correlation_matrix = cleaned_data[correlation_columns].corr()

display(correlation_matrix.round(3))


## 11. IQR outlier analysis

This is an **inspection only**. We do not remove statistical outliers because Isolation Forest is designed to identify unusual observations.


In [ ]:
outlier_results = []

for column in ["amount", "oldbalanceOrg", "newbalanceOrig"]:
    q1 = cleaned_data[column].quantile(0.25)
    q3 = cleaned_data[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (cleaned_data[column] < lower_bound)
        | (cleaned_data[column] > upper_bound)
    ).sum()

    outlier_results.append({
        "Feature": column,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "Outliers": outlier_count,
        "OutlierPercentage": outlier_count / len(cleaned_data) * 100,
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary.round(2))


## 12. Fraud transactions by type


In [ ]:
fraud_only = cleaned_data[cleaned_data["isFraud"] == 1].copy()

fraud_by_type = (
    fraud_only["type"]
    .value_counts()
    .to_frame("FraudTransactions")
)

fraud_by_type["Percentage"] = (
    fraud_by_type["FraudTransactions"]
    / len(fraud_only)
    * 100
)

display(fraud_by_type.round(2))


## 13. Inspect known fraudulent transactions


In [ ]:
display(
    fraud_only[
        [
            "type",
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "originBalanceChange",
            "amountToBalanceRatio",
        ]
    ].head(20)
)


# EDA-Informed Feature Engineering

The baseline project uses:

- `type`
- `amount`
- `oldbalanceOrg`
- `newbalanceOrig`

EDA suggests that additional behavioral relationships may help Isolation Forest detect unusual transactions more effectively.


In [ ]:
feature_data = cleaned_data.copy()

feature_data["originBalanceChange"] = (
    feature_data["oldbalanceOrg"]
    - feature_data["newbalanceOrig"]
)

feature_data["originEmptied"] = (
    feature_data["newbalanceOrig"] == 0
).astype(int)

feature_data["amountToBalanceRatio"] = np.where(
    feature_data["oldbalanceOrg"] > 0,
    feature_data["amount"] / feature_data["oldbalanceOrg"],
    0,
)

feature_data["originBalanceError"] = np.abs(
    (
        feature_data["oldbalanceOrg"]
        - feature_data["amount"]
    )
    - feature_data["newbalanceOrig"]
)

baseline_features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
]

improved_features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "originBalanceChange",
    "originEmptied",
    "amountToBalanceRatio",
    "originBalanceError",
]

baseline_data = feature_data[baseline_features].copy()
improved_data = feature_data[improved_features].copy()
fraud_labels = feature_data["isFraud"].copy()

print("Baseline features:")
print(baseline_features)

print("\nEDA-informed features:")
print(improved_features)

print("\nBaseline shape:", baseline_data.shape)
print("Improved shape:", improved_data.shape)


## 14. Engineered-feature behavior


In [ ]:
engineered_comparison = (
    feature_data
    .groupby("isFraud")
    .agg(
        AvgBalanceChange=("originBalanceChange", "mean"),
        MedianBalanceChange=("originBalanceChange", "median"),
        AvgAmountBalanceRatio=("amountToBalanceRatio", "mean"),
        MedianAmountBalanceRatio=("amountToBalanceRatio", "median"),
        AvgBalanceError=("originBalanceError", "mean"),
        MedianBalanceError=("originBalanceError", "median"),
        OriginEmptiedPercentage=("originEmptied", lambda x: x.mean() * 100),
    )
)

display(engineered_comparison.round(4))

print("\nNaN values in improved data:", improved_data.isnull().sum().sum())

numeric_improved = improved_data.select_dtypes(include=np.number)

print(
    "Infinite values in improved data:",
    np.isinf(numeric_improved.to_numpy()).sum()
)


# Key EDA Findings

After running the notebook, record the main findings here. From the full PaySim analysis already performed in this project, the important patterns include:

- The dataset contains **6,362,620 transactions**.
- Known fraud is very rare: approximately **0.1291%** of transactions.
- Fraud appears in **TRANSFER** and **CASH_OUT** transactions in this dataset.
- Fraudulent transactions have substantially higher typical transaction amounts than normal transactions.
- Approximately **98% of known fraudulent transactions leave the origin balance at zero**.
- Many known fraud transactions have an `amountToBalanceRatio` close to **1**, meaning the transaction amount is approximately the sender's available balance.
- `oldbalanceOrg` and `newbalanceOrig` are extremely highly correlated.
- Statistical outliers are retained because they may represent the unusual behavior Isolation Forest is intended to detect.
- EDA supports comparing a four-feature baseline model against an EDA-informed model with additional behavioral features.

## Next stage

The next project stage should move preprocessing, encoding, scaling, train/test splitting, Isolation Forest training, and evaluation into reusable Python files under `src/`.
